# qdmpy ODMR Spectral Models Tutorial

qdmpy provides three built-in spectral models for fitting NV-center ODMR data:

| Model | Isotope | Dips | Hyperfine |
|-------|---------|------|-----------|
| `ESR14N` | ¹⁴N (natural diamond) | 3 | 2.16 MHz |
| `ESR15N` | ¹⁵N (enriched diamond) | 2 | 3.03 MHz |
| `ESRSINGLE` | Any (broad or simple) | 1 | — |

All models are implemented as vectorised NumPy functions and registered in `ModelRegistry`.
They are callable via `model.func(freq_ghz, params)` and are used internally by `FitManager`.

This tutorial shows how to:
1. Explore models via `ModelRegistry`
2. Generate synthetic spectra with each model
3. Understand parameter layout
4. Use models in a fitting workflow

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from qdmpy.fitting import ESR14N, FitManager, ModelRegistry

## 1. ModelRegistry

In [ ]:
all_models = ModelRegistry.all()

for model_name in ["ESRSINGLE", "ESR15N", "ESR14N"]:
    model = ModelRegistry.get(model_name)

## 2. Model function signature

```python
spectra = model.func(freq_ghz, params)
```

- `freq_ghz` — 1-D frequency array in GHz, shape `(n_freq,)`
- `params` — 2-D parameter array, shape `(N, n_parameters)` for N pixels
- Returns `(N, n_freq)` array of synthetic spectra

For a single spectrum use `params.reshape(1, -1)` and squeeze the result.

In [ ]:
# Frequency axis around 2.87 GHz ZFS
freq = np.linspace(2.84, 2.90, 500)   # GHz

# ---------- ESRSINGLE ----------
# parameter_names: [center, width, contrast, offset]
model_s = ModelRegistry.get("ESRSINGLE")
params_s = np.array([[2.870, 0.005, 0.10, 0.0]])    # shape (1, 4)
spec_s = model_s.func(freq, params_s).squeeze()

# ---------- ESR15N ----------
# parameter_names: [center, width, contrast_0, contrast_1, offset]
model_15 = ModelRegistry.get("ESR15N")
params_15 = np.array([[2.870, 0.004, 0.08, 0.08, 0.0]])  # shape (1, 5)
spec_15 = model_15.func(freq, params_15).squeeze()

# ---------- ESR14N ----------
# parameter_names: [center, width, contrast_0, contrast_1, contrast_2, offset]
model_14 = ModelRegistry.get("ESR14N")
params_14 = np.array([[2.870, 0.004, 0.05, 0.05, 0.05, 0.0]])  # shape (1, 6)
spec_14 = model_14.func(freq, params_14).squeeze()

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axes[0].plot(freq, spec_s, color="C0", lw=2)
axes[0].set_title("ESRSINGLE — single Lorentzian dip")

axes[1].plot(freq, spec_15, color="C1", lw=2)
axes[1].set_title("ESR15N — \u00b9\u2075N doublet (I = \u00bd)")

axes[2].plot(freq, spec_14, color="C2", lw=2)
axes[2].set_title("ESR14N — \u00b9\u2074N triplet (I = 1)")
axes[2].set_xlabel("Frequency (GHz)")

for ax in axes:
    ax.set_ylabel("Normalised fluorescence")
    ax.grid(True, alpha=0.3)

plt.suptitle("ODMR spectral models \u2014 synthetic spectra", y=1.01)
plt.tight_layout()
plt.show()

## 3. Parameter layout

All models share the same parameter slot ordering as pyGpufit.  The `parameter_names`
list matches the column order in the parameter array.

In [ ]:
for model_name in ["ESRSINGLE", "ESR15N", "ESR14N"]:
    model = ModelRegistry.get(model_name)

## 4. Vectorised evaluation — many pixels at once

The model functions are fully vectorised: passing `N` parameter rows evaluates
`N` spectra simultaneously with no Python loop.

In [ ]:
rng = np.random.default_rng(42)
N = 1000

# Random ESR14N parameters: [center, width, contrast_0, contrast_1, contrast_2, offset]
centers = rng.uniform(2.865, 2.875, N)
widths = rng.uniform(0.003, 0.006, N)
contrasts = rng.uniform(0.03, 0.08, N)

params = np.column_stack([
    centers,
    widths,
    contrasts, contrasts, contrasts,  # equal contrast per dip
    np.zeros(N),                       # offset
]).astype(np.float32)  # shape (N, 6)

spectra = model_14.func(freq, params)  # shape (N, len(freq))

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(min(20, N)):
    ax.plot(freq, spectra[i], alpha=0.3, lw=1, color="C2")
ax.plot(freq, spectra.mean(axis=0), "k-", lw=2, label="Mean")
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Normalised fluorescence")
ax.set_title(f"ESR14N \u2014 {N} random pixel spectra")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Model selection guide

| Condition | Choose |
|-----------|--------|
| Natural diamond (¹⁴N), resolved hyperfine | `ESR14N` |
| Isotopically enriched ¹⁵N diamond | `ESR15N` |
| Hyperfine unresolved (broad lines, high strain) | `ESRSINGLE` |
| Unsure | `FitManager("auto")` — auto-detects from data |

## 6. Integration with FitManager

Pass the model name string to `FitManager` — it looks up the model via `ModelRegistry`
internally.

In [ ]:
# Both of these are equivalent:
fitm_a = FitManager("ESR14N")
fitm_b = FitManager(ESR14N.name)   # ESR14N.name == "ESR14N"


## Summary

| Class / function | Purpose |
|------------------|---------|
| `ModelRegistry.all()` | Dict of registered model classes |
| `ModelRegistry.get(name)` | Retrieve a model instance by name |
| `model.func(freq, params)` | Evaluate `N` spectra vectorised |
| `model.parameter_names` | Ordered list of parameter slot names |
| `model.n_parameters` | Total parameter count |
| `model.units` | Dict mapping parameter name → unit string |
| `FitManager(model_name)` | Use a model for GPU fitting |